# Práctica 7 · Clasificación de textos
### TF-IDF vs. CountVectorizer + Regresión Logística

**Objetivo:** comparar cómo cambia la clasificación cuando representamos los mismos documentos con TF-IDF o con conteos de palabras.

Basado en **Clase 20**, diapositivas 29–44. Se usan las cuatro categorías originales de *20 Newsgroups*. Las explicaciones están en español y los ejemplos de texto en inglés.

## 1. Librerías y datos

Entrenaremos dos clasificadores con **exactamente los mismos documentos**. Usaremos las etiquetas originales solo para aprendizaje supervisado y evaluación.

> Ejecutar en Google Colab o Jupyter con Internet: la primera carga del dataset puede descargar archivos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

categorias = [
    "sci.space", "rec.sport.baseball", "comp.graphics", "talk.politics.mideast"
]

datos = fetch_20newsgroups(
    subset="train", categories=categorias,
    remove=("headers", "footers", "quotes"),
    shuffle=True, random_state=42
)
documentos = datos.data
etiquetas = datos.target
nombres = datos.target_names

print("Documentos:", len(documentos))
print("Categorías:", nombres)

In [ ]:
i = 11
print("Categoría:", nombres[etiquetas[i]])
print("Texto:\n", documentos[i][:500])

## 2. Entrenamiento y prueba

Usamos **80 %** de los documentos para entrenar y **20 %** para evaluar. `stratify` conserva aproximadamente la proporción de cada categoría.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    documentos, etiquetas,
    test_size=0.2, random_state=42, stratify=etiquetas
)
print("Entrenamiento:", len(X_train))
print("Prueba:", len(X_test))

## 3. Dos pipelines: TF-IDF y CountVectorizer

- **TF-IDF:** pondera palabras según frecuencia y rareza; además normaliza con L2.
- **CountVectorizer:** cuenta apariciones. **No** utiliza `norm="l2"`, pues ese parámetro no existe en `CountVectorizer`.

El clasificador y sus hiperparámetros se mantienen iguales para que la comparación se concentre en la representación.

In [ ]:
parametros = dict(
    stop_words="english",
    min_df=5,
    max_df=0.8,
    max_features=5000
)

modelo_tfidf = Pipeline([
    ("vectorizador", TfidfVectorizer(**parametros, norm="l2")),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

modelo_count = Pipeline([
    ("vectorizador", CountVectorizer(**parametros)),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

modelos = {"TF-IDF": modelo_tfidf, "CountVectorizer": modelo_count}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    print(nombre, "→ vocabulario:", len(modelo.named_steps["vectorizador"].get_feature_names_out()))

## 4. Evaluación en los mismos documentos

- **Accuracy:** proporción de aciertos.
- **Precision:** aciertos entre los predichos en una clase.
- **Recall:** aciertos entre los verdaderos de una clase.
- **F1:** media armónica de precision y recall.

In [ ]:
predicciones = {}
resultados = []

for nombre, modelo in modelos.items():
    pred = modelo.predict(X_test)
    predicciones[nombre] = pred
    acc = accuracy_score(y_test, pred)
    resultados.append({"Representación": nombre, "Accuracy": acc})
    print(f"\n{nombre} — Accuracy: {acc:.3f}")
    print(classification_report(y_test, pred, target_names=nombres))

tabla_resultados = pd.DataFrame(resultados)
display(tabla_resultados.style.format({"Accuracy": "{:.3f}"}))

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.bar(tabla_resultados["Representación"], tabla_resultados["Accuracy"])
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Comparación de representaciones")
for i, valor in enumerate(tabla_resultados["Accuracy"]):
    plt.text(i, valor + 0.02, f"{valor:.3f}", ha="center")
plt.show()

### Matrices de confusión

La **diagonal** indica predicciones correctas. Fuera de ella están los errores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (nombre, pred) in zip(axes, predicciones.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=nombres,
        xticks_rotation=40, ax=ax, colorbar=False
    )
    ax.set_title(nombre)
fig.tight_layout()
plt.show()

## 5. Examinar aciertos y errores

Mostramos ejemplos de ambos modelos. Los errores ayudan a detectar textos ambiguos, demasiado cortos o con vocabulario compartido.

In [ ]:
def mostrar_ejemplos(nombre, aciertos=True, cantidad=3):
    pred = predicciones[nombre]
    indices = np.where(pred == y_test if aciertos else pred != y_test)[0]
    print(f"{nombre}: {len(indices)} {'aciertos' if aciertos else 'errores'}")
    for i in indices[:cantidad]:
        print("\n---")
        print("Real:", nombres[y_test[i]])
        print("Predicción:", nombres[pred[i]])
        print(X_test[i][:350].replace("\n", " "))

for nombre in modelos:
    mostrar_ejemplos(nombre, aciertos=True, cantidad=2)
    mostrar_ejemplos(nombre, aciertos=False, cantidad=2)

## 6. Coeficientes: palabras más influyentes

En una clasificación con cuatro categorías, cada clase tiene su propia fila de coeficientes. Un peso **positivo** favorece esa categoría frente a las demás; uno **negativo** la desfavorece. Comparamos los diez extremos de cada fila.

In [ ]:
def palabras_influyentes(modelo, cantidad=10):
    terminos = modelo.named_steps["vectorizador"].get_feature_names_out()
    clf = modelo.named_steps["clasificador"]
    filas = []
    for fila, clase in enumerate(clf.classes_):
        pesos = clf.coef_[fila]
        positivos = np.argsort(pesos)[-cantidad:][::-1]
        negativos = np.argsort(pesos)[:cantidad]
        filas.append({
            "Categoría": nombres[clase],
            "Positivas (+)": ", ".join(terminos[positivos]),
            "Negativas (−)": ", ".join(terminos[negativos])
        })
    return pd.DataFrame(filas)

for nombre, modelo in modelos.items():
    print("\n", nombre)
    display(palabras_influyentes(modelo))

In [ ]:
# Una gráfica sencilla para comparar los coeficientes por categoría.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (nombre, modelo) in zip(axes, modelos.items()):
    vectorizador = modelo.named_steps["vectorizador"]
    clf = modelo.named_steps["clasificador"]
    # Tomamos la primera categoría, solo para una visualización legible.
    pesos = clf.coef_[0]
    orden = np.argsort(np.abs(pesos))[-12:]
    palabras = vectorizador.get_feature_names_out()[orden]
    ax.barh(palabras, pesos[orden])
    ax.set_title(f"{nombre}: {nombres[clf.classes_[0]]}")
    ax.set_xlabel("Coeficiente de regresión logística")
fig.tight_layout()
plt.show()

## 7. Una noticia real en inglés

**Noticia:** “Brewers sweep the Dodgers in the NLCS to reach the World Series for the first time since 1982”  
**Fuente:** Associated Press (Beth Harris), **15 de octubre de 2026**.  
**Enlace:** https://apnews.com/article/brewers-dodgers-score-nlcs-cfd30befcec42a035cbea5cae3bf746c

Para respetar los derechos de autor, el texto siguiente es un **resumen original en inglés de la noticia**, no un fragmento copiado de AP. El evento real corresponde a la categoría esperada `rec.sport.baseball`. Puedes sustituir `noticia` por un fragmento breve del artículo para comparar.

In [ ]:
titulo_noticia = "Brewers sweep the Dodgers in the NLCS to reach the World Series for the first time since 1982"
url_noticia = "https://apnews.com/article/brewers-dodgers-score-nlcs-cfd30befcec42a035cbea5cae3bf746c"

noticia = """
The Milwaukee Brewers advanced to the World Series after defeating the Los Angeles
Dodgers in four games of the National League Championship Series. In the final
baseball game, Milwaukee won 6-4 after overcoming an early Dodgers lead.
Garrett Mitchell hit a go-ahead home run in the sixth inning, while the Brewers'
bullpen and relief pitchers protected the lead. Shohei Ohtani homered for Los
Angeles, but Milwaukee completed the playoff sweep. The Brewers finished the
regular season with 103 victories and will return to the World Series for the
first time since 1982. Baseball fans celebrated the National League pennant.
""".strip()

print(titulo_noticia)
print(url_noticia)
print("\nTexto analizado (resumen original):\n", noticia)

### Clasificación y probabilidades

Ambos modelos ven **exactamente el mismo texto**. `predict_proba` estima la distribución entre las cuatro categorías posibles; son probabilidades del modelo, no garantías de corrección.

In [ ]:
filas = []
for nombre, modelo in modelos.items():
    etiqueta = modelo.predict([noticia])[0]
    clf = modelo.named_steps["clasificador"]
    probabilidades = modelo.predict_proba([noticia])[0]
    print(f"{nombre} → categoría predicha: {nombres[etiqueta]}")
    for clase, prob in zip(clf.classes_, probabilidades):
        filas.append({"Representación": nombre, "Categoría": nombres[clase], "Probabilidad": prob})

tabla_noticia = pd.DataFrame(filas)
display(tabla_noticia.pivot(index="Categoría", columns="Representación", values="Probabilidad").round(3))

In [ ]:
comparacion = tabla_noticia.pivot(
    index="Categoría", columns="Representación", values="Probabilidad"
).reindex(nombres)

comparacion.plot(kind="bar", figsize=(10, 4))
plt.ylabel("Probabilidad estimada")
plt.xlabel("Categoría")
plt.ylim(0, 1)
plt.title("Clasificación de la noticia: TF-IDF vs. CountVectorizer")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 8. Discusión y conclusión

Observa las métricas, las palabras y las predicciones anteriores. **No compares directamente la magnitud de los coeficientes** entre representaciones: sus escalas de entrada son diferentes.

In [ ]:
acc_tfidf = tabla_resultados.loc[tabla_resultados["Representación"] == "TF-IDF", "Accuracy"].iloc[0]
acc_count = tabla_resultados.loc[tabla_resultados["Representación"] == "CountVectorizer", "Accuracy"].iloc[0]

ganador = "TF-IDF" if acc_tfidf > acc_count else "CountVectorizer" if acc_count > acc_tfidf else "Empate"
print(f"Accuracy TF-IDF:        {acc_tfidf:.3f}")
print(f"Accuracy CountVectorizer: {acc_count:.3f}")
print("Mejor accuracy:", ganador)
print("Diferencia absoluta:", f"{abs(acc_tfidf - acc_count):.3f}")

for nombre, modelo in modelos.items():
    etiqueta = modelo.predict([noticia])[0]
    prob = modelo.predict_proba([noticia]).max()
    print(f"Noticia ({nombre}): {nombres[etiqueta]} — {prob:.3f}")

**Responde brevemente después de ejecutar:**

1. ¿Qué representación obtuvo mayor accuracy y cuál tuvo mejores valores F1 por categoría?
2. ¿Qué confusiones aparecen en las matrices? ¿Coinciden entre modelos?
3. ¿Qué palabras positivas y negativas explican mejor alguna categoría?
4. ¿Qué modelos clasificaron correctamente la noticia de béisbol? Compara sus probabilidades.
5. ¿Por qué los conteos brutos podrían dar más importancia a documentos largos que TF-IDF normalizado?

**Conclusión:** la representación modifica las características de entrada y, por tanto, las decisiones aprendidas por el clasificador. La mejor opción se decide con las métricas del conjunto de prueba, no solamente con una noticia.

# R3 visualization

**Visualización adicional (interactiva).** Reducimos a tres dimensiones con `TruncatedSVD`, una alternativa a PCA apropiada para matrices dispersas. Puedes rotar y acercar los puntos.

Los botones muestran: categorías reales, categorías predichas, aciertos/errores, **esfera** (normalización L2 en 3D) y **vecinos de la noticia** (15 documentos más cercanos por coseno en el espacio original).

> Los modelos se entrenaron en la representación original, **no** en 3D. La proyección altera distancias: solo se utiliza para explorar.

In [ ]:
# Si Plotly no está instalado: %pip install plotly
import plotly.graph_objects as go
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

colores = ["#3366cc", "#e66a35", "#198754", "#8e44ad"]

def normalizar_puntos(x):
    return normalize(x, norm="l2")

# Limitamos puntos para que sea fácil rotar en Colab/Jupyter.
indices_3d = np.random.default_rng(42).choice(
    len(X_test), size=min(450, len(X_test)), replace=False
)

In [ ]:
def grafica_3d(nombre, modelo):
    vectorizador = modelo.named_steps["vectorizador"]
    X_3d = vectorizador.transform([X_test[i] for i in indices_3d])
    X_noticia = vectorizador.transform([noticia])

    svd = TruncatedSVD(n_components=3, random_state=42)
    puntos = svd.fit_transform(X_3d)
    puntos_esfera = normalizar_puntos(puntos)
    punto_noticia = svd.transform(X_noticia)

    reales = np.asarray(y_test)[indices_3d]
    preds = predicciones[nombre][indices_3d]
    vecinos = np.argsort(cosine_similarity(X_3d, X_noticia).ravel())[-15:]
    textos = [X_test[i][:80].replace("\n", " ") for i in indices_3d]
    hover = [f"Real: {nombres[r]}<br>Pred: {nombres[p]}<br>{t}" for r,p,t in zip(reales,preds,textos)]

    def nube(x, color, etiqueta, tam=4):
        return go.Scatter3d(
            x=x[:,0], y=x[:,1], z=x[:,2], mode="markers",
            marker=dict(size=tam, color=color, opacity=0.75),
            text=hover, hovertemplate="%{text}<extra></extra>", name=etiqueta
        )

    colores_reales = [colores[k] for k in reales]
    colores_pred = [colores[k] for k in preds]
    colores_aciertos = ["#2ca25f" if r == p else "#e63946" for r,p in zip(reales,preds)]
    colores_vecinos = ["#ffae00" if i in vecinos else "#b9c0c9" for i in range(len(reales))]

    fig = go.Figure()
    fig.add_trace(nube(puntos, colores_reales, "Categoría real"))
    fig.add_trace(nube(puntos, colores_pred, "Predicción"))
    fig.add_trace(nube(puntos, colores_aciertos, "Correcto / error"))
    fig.add_trace(nube(puntos_esfera, colores_pred, "Proyección normalizada"))
    fig.add_trace(nube(puntos, colores_vecinos, "Vecinos de la noticia", tam=5))
    fig.add_trace(go.Scatter3d(
        x=punto_noticia[:,0], y=punto_noticia[:,1], z=punto_noticia[:,2],
        mode="markers", marker=dict(size=11, color="#df1c36", symbol="diamond"),
        name="Noticia", hovertext=titulo_noticia, hoverinfo="text"
    ))

    botones = []
    opciones = ["Reales", "Predichas", "Aciertos/errores", "Esfera", "Vecinos noticia"]
    for j, texto in enumerate(opciones):
        visible = [i == j for i in range(5)] + [j == 4]
        botones.append(dict(
            label=texto, method="update", args=[{"visible":visible}, {"title":f"{nombre} · {texto}"}]
        ))
    fig.update_traces(visible=False)
    fig.data[0].visible = True
    fig.update_layout(
        title=f"{nombre} · Reales", height=610,
        margin=dict(l=0,r=0,b=0,t=90),
        updatemenus=[dict(type="buttons",direction="right",buttons=botones,x=0,y=1.13)],
        scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
        showlegend=False
    )
    fig.show()

for nombre, modelo in modelos.items():
    grafica_3d(nombre, modelo)

**Lectura de la visualización:** los colores de “Predichas” corresponden a las cuatro categorías, en el mismo orden mostrado en el dataset. En “Aciertos/errores”, verde = acierto y rojo = error. En “Vecinos noticia”, amarillo = uno de los 15 vecinos por similitud coseno en la representación de ese modelo y el rombo rojo = noticia nueva. La posición del rombo es solo su proyección 3D.

La opción **Esfera** normaliza los vectores *después* de reducir a tres dimensiones: no equivale a normalizar los datos originales antes de entrenar ni transforma la regresión logística.